<a href="https://colab.research.google.com/github/vshalisko/GEE/blob/main/Colab/TerraClimate_prec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
import ee
import geemap

# 1. Autenticación e inicialización de Google Earth Engine
ee.Authenticate()
ee.Initialize(project='ee-vshalisko')

# Cargar la colección
dataset = ee.ImageCollection('IDAHO_EPSCOR/TERRACLIMATE')
dataset = dataset.select(['pr'])

# Verificación: Imprimir el número de imágenes mensuales filtradas
print('Número de imágenes mensuales:', dataset.size().getInfo())

# Calcular la precipitación total en el periodo elegido para cada año
years = ee.List.sequence(1998, 2002) # Lista de años

def calculate_yearly_sum(year, mes_inicio, mes_fin):
    ## parametros de la funcion:
    ## secuencia de años ee.List
    ## mes de inicio, mes de final

    ## Diccionario para almacenar el número de días en cada mes
    ## (asumiendo un año no bisiesto para febrero)
    days_in_month = {
        1: 31, 2: 28, 3: 31, 4: 30, 5: 31, 6: 30,
        7: 31, 8: 31, 9: 30, 10: 31, 11: 30, 12: 31
    }

    # Obtener el número de días para el mes de fin.
    # Usamos .get(mes_fin, 28) para tener un valor por defecto
    num_days_end_month = days_in_month.get(mes_fin, 28)

    start_date = ee.Date.fromYMD(year, mes_inicio, 1)
    end_date = ee.Date.fromYMD(year, mes_fin, num_days_end_month)
    yearly_collection = dataset.filterDate(start_date, end_date)

    # Sumar la precipitación mensual en el periodo para este año específico
    yearly_sum = yearly_collection.reduce(ee.Reducer.sum()).rename('pr') # Asegurar que la banda se llama 'pr'
    return yearly_sum.set('year', year)

def calculate_yearly_sum_seca(year):
    # variante de la funcion para la temporada seca
    return calculate_yearly_sum(year, 3, 5)

def calculate_yearly_sum_humeda(year):
  # variante de la funcion para la temporada seca
    return calculate_yearly_sum(year, 8, 10)

# Mapear sobre los años para obtener una ImageCollection de las sumas anuales de precipitacion
yearly_seca_sums = ee.ImageCollection(years.map(calculate_yearly_sum_seca))

# Verificación: Imprimir el número de sumas anuales calculadas
print('Número de sumas anuales de precipitacion en el periodo seco de año:', yearly_seca_sums.size().getInfo())

precip_seca_mean = yearly_seca_sums.reduce(ee.Reducer.mean()).rename('precip_mean_seca')

# Mapear sobre los años para obtener una ImageCollection de las sumas anuales de precipitacion
yearly_humeda_sums = ee.ImageCollection(years.map(calculate_yearly_sum_humeda))

# Verificación: Imprimir el número de sumas anuales calculadas
print('Número de sumas anuales de precipitacion en el periodo humedo de año:', yearly_humeda_sums.size().getInfo())

precip_humeda_mean = yearly_humeda_sums.reduce(ee.Reducer.mean()).rename('precip_mean_humeda')

Número de imágenes mensuales: 804
Número de sumas anuales de precipitacion en el periodo seco de año: 5
Número de sumas anuales de precipitacion en el periodo humedo de año: 5


In [ ]:
# 6. Definir parámetros de visualización para la precipitación (en mm)
vis_params = {
    'min': 0,
    'max': 1500, # Ajustado de 5000 a 1500 para el promedio anual
    'palette': ['blue', 'limegreen', 'yellow', 'orange', 'red']
}

# 7. Crear el mapa interactivo y agregar la capa
Map = geemap.Map(center=[19.43, -99.13], zoom=4, basemap='OpenTopoMap') # Centrado por defecto en México, solo OpenTopoMap
Map.addLayer(precip_seca_mean, vis_params, 'Promedio Precipitación periodo seco (mm)')
Map.addLayer(precip_humeda_mean, vis_params, 'Promedio Precipitación periodo humedo (mm)')
Map.add_colorbar(vis_params, label="Precipitación Promedio (mm)")

# Mostrar mapa en Colab
Map